## Method1:增加随机扰动

In [6]:
import pandas as pd
import numpy as np
import random

# Step 1: 原始数据导入（建议保存为 CSV）
# 若你已复制为 CSV 文件，可以使用 pd.read_csv("your_file.csv")

data= pd.read_csv('/Users/evelyndu/Desktop/APP Design Ideas/Salary/data.csv')
df = pd.DataFrame(data)

# Step 2: 增加数据量
def generate_new_data(df, n_samples=2000):
    new_rows = []
    for _ in range(n_samples):
        row = df.sample(n=1).iloc[0]
        new_row = row.copy()

        # 添加微小随机扰动（浮动5%内）
        new_row["现金"] = round(row["现金"] * random.uniform(0.95, 1.05), 2)
        if row["期权"] != "":
            new_row["期权"] = round(float(row["期权"]) * random.uniform(0.9, 1.1), 3)
            new_row["总包"] = round(new_row["现金"] + new_row["期权"], 3)
        else:
            new_row["期权"] = ""
            new_row["总包"] = new_row["现金"]

        new_rows.append(new_row)

    return pd.DataFrame(new_rows)

# Step 3: 生成并导出
new_df = generate_new_data(df, n_samples=2000)
new_df.to_csv("/Users/evelyndu/Desktop/APP Design Ideas/Salary/extend_data.csv", index=False)
new_df.head()

,公司,岗位,职级,工作年限,年薪,学历水平,现金,期权,总包
13,字节跳动,产品,2.2,4,26,本,80.52,7.213,87.733
7,字节跳动,运营,2.2,3,25,本,47.24,NaN,NaN
38,字节跳动,产品,3.1,9,33,本,131.27,25.597,156.867
39,字节跳动,产品,3.1,11,33,硕,116.01,9.179,125.189
26,字节跳动,运营,2.2,6,28,本,60.09,NaN,NaN


## Method2:通过高级模拟

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KernelDensity
import random

# 读取原始数据 DataFrame
df = pd.read_excel("原始样本.xlsx")  # 替换为你真实路径

# 去除空值或非数值处理
df = df[df['现金'].notna()]
df['期权'] = df['期权'].replace('', 0).astype(float)

# 提取连续变量用于 KDE 拟合
features = df[['现金', '期权']].values
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# KDE 拟合联合概率密度（现金 + 期权）
kde = KernelDensity(kernel='gaussian', bandwidth=0.4)
kde.fit(features_scaled)

# 采样：从KDE中进行蒙特卡洛采样
n_samples = 2000
samples_scaled = kde.sample(n_samples=n_samples, random_state=42)
samples = scaler.inverse_transform(samples_scaled)

# 从原始数据中随机采样离散字段（岗位/职级/学历/工作年限等）
categorical_cols = ['岗位', '职级', '工作年限', '年薪', '学历水平']
cat_samples = df[categorical_cols].sample(n=n_samples, replace=True).reset_index(drop=True)

# 构建新 DataFrame
new_df = cat_samples.copy()
new_df['公司'] = '字节跳动'
new_df['现金'] = np.round(samples[:, 0], 2)
new_df['期权'] = np.round(samples[:, 1], 3)
new_df['总包'] = np.round(new_df['现金'] + new_df['期权'], 3)

# 修正异常（防止负值）
new_df['现金'] = new_df['现金'].apply(lambda x: max(x, 15))
new_df['期权'] = new_df['期权'].apply(lambda x: max(x, 0))
new_df['总包'] = new_df['现金'] + new_df['期权']

# 调整列顺序
cols_order = ['公司', '岗位', '职级', '工作年限', '年薪', '学历水平', '现金', '期权', '总包']
new_df = new_df[cols_order]

# 导出为 Excel
new_df.to_excel("高级模拟_字节薪资2000行_MonteCarlo.xlsx", index=False)

# 查看前几行
new_df.head()